# Week 4 Capstone — Sales Data Analysis

EDA, data cleaning, regression model, visualizations and recommendations.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

raw = pd.read_csv('../data/Sample-Superstore.csv', dtype=str)
valid = raw.iloc[:9994].copy()
returns = raw.iloc[9994:].copy()

valid['Order Date'] = pd.to_datetime(valid['Order Date'], errors='coerce')
valid['Ship Date'] = pd.to_datetime(valid['Ship Date'], errors='coerce')
for c in ['Sales','Quantity','Discount','Profit','Postal Code']:
    valid[c] = pd.to_numeric(valid[c], errors='coerce')
valid['Year'] = valid['Order Date'].dt.year
valid['Month'] = valid['Order Date'].dt.month
valid['Quarter'] = 'Q' + valid['Order Date'].dt.quarter.astype('Int64').astype(str)
valid['Returned'] = valid['Order ID'].isin(returns['Order ID'].dropna()).astype(int)
valid = valid.drop_duplicates().dropna(subset=['Sales','Order Date'])
print(valid.shape)

## EDA

In [ ]:
print('Total sales:', valid.Sales.sum())
print('Total profit:', valid.Profit.sum())
print('Orders:', valid['Order ID'].nunique())
print('Customers:', valid['Customer ID'].nunique())
display(valid.groupby('Category').agg(Sales=('Sales','sum'), Profit=('Profit','sum')).sort_values('Sales', ascending=False))
display(valid.groupby('Region').agg(Sales=('Sales','sum'), Profit=('Profit','sum')).sort_values('Profit', ascending=False))

In [ ]:
monthly = valid.assign(MonthPeriod=valid['Order Date'].dt.to_period('M')).groupby('MonthPeriod').agg(Sales=('Sales','sum'), Profit=('Profit','sum'))
monthly.plot(y='Sales', figsize=(10,4), marker='o', title='Monthly Sales')
plt.xticks(rotation=60)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7,4))
plt.scatter(valid['Discount'], valid['Profit'], alpha=0.25)
plt.axhline(0, linewidth=1)
plt.title('Discount vs Profit')
plt.xlabel('Discount')
plt.ylabel('Profit')
plt.tight_layout()
plt.show()

## Regression model

Time-based holdout: 2015–2017 training and 2018 testing. Profit is excluded to avoid target leakage.

In [ ]:
features = ['Quantity','Discount','Year','Month','Quarter','Ship Mode','Segment','Region','Category','Sub-Category']
train = valid[valid.Year < 2018]
test = valid[valid.Year == 2018]
cat_cols = [c for c in features if train[c].dtype == 'object']
num_cols = [c for c in features if c not in cat_cols]

pre = ColumnTransformer([
    ('num', SimpleImputer(strategy='median'), num_cols),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ]), cat_cols)
])
model = RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1, min_samples_leaf=2, max_features='sqrt')
pipe = Pipeline([('pre',pre),('model',model)])
pipe.fit(train[features], train.Sales)
pred = pipe.predict(test[features])

mae = mean_absolute_error(test.Sales, pred)
rmse = mean_squared_error(test.Sales, pred)**0.5
r2 = r2_score(test.Sales, pred)
print({'MAE':mae,'RMSE':rmse,'R2':r2})

## Recommendations

1. Protect Technology category momentum with inventory and cross-sell planning.
2. Review pricing and discounting for weak-profit sub-categories, especially Tables.
3. Use discount guardrails because high discounts are associated with weaker profitability.
4. Use monthly trends for peak-season inventory and staffing.
5. Add marketing, customer-history and inventory variables to improve future sales forecasts.